# Inference Bioinspired123D - Agentic System for 3D Bioinspired Design
Rachel K. Luu, 2026

In [ ]:
import os
import sys, io, os, signal
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout
from scripts.inference.state import DesignState
from scripts.inference.config import Bio3DConfig, RAGConfig
from scripts.inference.llm_bio3d import Bio3D
from scripts.inference.openai_client import OpenAIClient, OpenAIConfig
from scripts.inference.blender_exec import BlenderValidator, BlenderExecConfig
from scripts.inference.vlm_text_rag import VLMTextRag, VLMRagConfig
from scripts.inference.vlm_critic import VLMCritic, VLMCriticConfig
from scripts.inference.nodes import Bio3DNode, VLMNode, CodeFixerNode, CodeDesignerNode, NodeConfig
from scripts.inference.graph_agent import build_bio3d_agent
from scripts.inference.report_pdf import generate_run_report, generate_detailed_report  # optional


os.environ["OPENAI_API_KEY"] = "YOUR TOKEN HERE"

# Load Agentic Graph

In [ ]:
WSL_RENDER_BASE = "/home/rachel/bio3d_renders" # CHANGE THIS TO WHERE YOU WANT ALL OUTPUT FILES TO GO

bio3d = Bio3D(
    Bio3DConfig(
        device="cuda:0",
        base_model="meta-llama/Llama-3.2-3B-Instruct",
        lora_adapter="rachelkluu/bioinspired3D",
        torch_dtype="float16",
    ),
    RAGConfig(
        enabled=True,
        jsonl_paths=["../data/rag/biodataset_RAG.jsonl"],
        embed_model_name="BAAI/bge-small-en-v1.5",
        top_k=2,
    ),
).load()

blender = BlenderValidator(BlenderExecConfig(
    blender_path="/mnt/c/Users/Rachel/Downloads/blender-4.2.1-windows-x64/blender-4.2.1-windows-x64/blender.exe",
    win_render_base=r"C:\Users\Rachel\bio3d_tmp\renders",
    wsl_render_base=WSL_RENDER_BASE,
))

oa = OpenAIClient(OpenAIConfig()) 

vlm_rag = VLMTextRag(VLMRagConfig(
    jsonl_path="../data/rag/biodataset_VLM_RAG.jsonl",
    embed_model_name="BAAI/bge-small-en-v1.5",
    top_k=2,
)).load()

critic = VLMCritic(oa, vlm_rag, VLMCriticConfig(top_k_refs=2))

node_cfg = NodeConfig(
    debug_dir="/home/rachel/bio3d_debug_fixes",
    max_code_fix_attempts=3,
    max_design_fix_attempts=1,
)

node_bio3d = Bio3DNode(bio3d, blender)
node_vlm = VLMNode(critic)
node_codefixer = CodeFixerNode(oa, blender, cfg=node_cfg, bio3d_for_rag=bio3d)
node_codedesigner = CodeDesignerNode(oa, blender, cfg=node_cfg, bio3d_for_rag=bio3d)

bio3d_agent = build_bio3d_agent(node_bio3d, node_vlm, node_codefixer, node_codedesigner)


# Inference Agentic Graph through the Initial State

In [ ]:
# -----------------------
# Initial state
# -----------------------
initial_state: DesignState = {
    "design_prompt": "tubular structure with cortical layers", #### CHANGE THIS DESIGN PROMPT HERE
    "approved": False,
    "fix_attempts_used": 0,
    "fix_success": False,
    "iteration_count": 0,
    "render_subdir_wsl": os.path.join(WSL_RENDER_BASE, datetime.now().strftime("%Y-%m-%d_%H-%M-%S")),
}


class TeeStream:
    def __init__(self):
        self.buffer = io.StringIO()
        self._stdout = sys.stdout

    def write(self, text):
        self._stdout.write(text)
        self.buffer.write(text)
        self._stdout.flush()
        self.buffer.flush()

    def flush(self):
        self._stdout.flush()
        self.buffer.flush()

    def getvalue(self):
        return self.buffer.getvalue()

class TimeoutException(Exception):
    pass

def _handler(signum, frame):
    raise TimeoutException("⏰ Timeout: run exceeded limit")

signal.signal(signal.SIGALRM, _handler)

timeout_s = 1800  # 30 min
signal.alarm(timeout_s)

print("Running Bioinspired123D pipeline...")

tee = TeeStream()
result = None

try:
    with redirect_stdout(tee):
        result = bio3d_agent.invoke(initial_state)
except TimeoutException as e:
    print(str(e))
    result = dict(initial_state)
    result["approved"] = False
    result["final_result"] = result.get("render_path")
    result["blender_status"] = "timeout"
finally:
    signal.alarm(0)

pipeline_log = tee.getvalue()

print("\n✅ FINAL SUMMARY:")
print("Approved:", result.get("approved"))
print("Blender status:", result.get("blender_status"))
print("Final render:", result.get("final_result"))

# -----------------------
# Optional: PDF report
# -----------------------
make_pdf = True
make_detailed_pdf = False

if make_pdf:
    generate_run_report(result, pipeline_log, wsl_render_base=WSL_RENDER_BASE, debug_dir=node_cfg.debug_dir)

if make_detailed_pdf:
    generate_detailed_report(result, pipeline_log, wsl_render_base=WSL_RENDER_BASE)

# Inference BioinspiredLLM to Establish Initial State

In [ ]:
from scripts.inference.llm_biollm import BioinspiredLLM

bio_llm = BioinspiredLLM(
    model_url=(
        "https://huggingface.co/rachelkluu/"
        "Llama3.1-8b-Instruct-CPT-SFT-DPO-09022024-Q4_K_M-GGUF/"
        "resolve/main/llama3.1-8b-instruct-cpt-sft-dpo-09022024-q4_k_m.gguf"
    ),
    main_gpu=1, 
).load()

design_prompt = bio_llm.generate_design_concept("horse hoof wall")  # INSERT BIOLOGICAL MATERIAL HERE
print("🧩 Generated design concept:", design_prompt)

initial_state: DesignState = {
    "design_prompt": design_prompt,
    "approved": False,
    "fix_attempts_used": 0,
    "fix_success": False,
    "iteration_count": 0,
    "render_subdir_wsl": os.path.join(
        WSL_RENDER_BASE,
        datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
    ),
}

result = bio3d_agent.invoke(initial_state)
